In [1]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


default_proxy_config = {
    'http': 'http://127.0.0.1:7890',
    'https': 'http://127.0.0.1:7890',
    'all': 'socks5://127.0.0.1:7890',
}


# default_proxy_config = {
#     'http': 'http://10.176.52.116:7890',
#     'https': 'http://10.176.52.116:7890',
#     'all': 'socks5://10.176.52.116:7890',
# }


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [2]:
import requests
resp = requests.get('https://papers.cool/venue/ICLR.2024?group=Poster&show=2000', proxies=default_proxy_config)
print(resp)
print(resp.headers['content-type'])
with open('iclr2024_2000.html', 'wb') as f:
    f.write(resp.content)

<Response [200]>
text/html; charset=UTF-8


In [5]:
from RFC.item.item import ItemType
from RFC.utils.parse import (
    get_html_soup,
    parse,
)


def parse_kimi_links(resp):
    parse_config = {
        'kimi_links': {
            ('attr', 'a', 'class', 'title-kimi', None): {}
        },
        'pdf_links': {
            ('attr', 'a', 'class', 'title-pdf', None): {}
        },
        'title': {
            ('attr', 'a', 'class', 'title-link', None): {}
        }
    }
    
    # kimi-links
    kimi_link_template = 'https://papers.cool/venue/kimi?paper={}'
    kimi_links_soup = parse(get_html_soup(resp.content), parse_config['kimi_links'])
    kimi_links_id = [soup['id'] for soup in kimi_links_soup]
    kimi_links = []
    for kimi_link_id in kimi_links_id:
        assert kimi_link_id.startswith('kimi-')
        kimi_links.append(kimi_link_template.format(kimi_link_id[len('kimi-'):]))
    
    # pdf-links
    pdf_links_soup = parse(get_html_soup(resp.content), parse_config['pdf_links'])
    raw_pdf_links = [soup['onclick'].split(', ')[1] for soup in pdf_links_soup]
    pdf_links = []
    for raw_pdf_link in raw_pdf_links:
        if raw_pdf_link.startswith('\''):
            raw_pdf_link = raw_pdf_link[1:]
        if raw_pdf_link.endswith('\''):
            raw_pdf_link = raw_pdf_link[:-1]
        if raw_pdf_link.startswith('/pdf?url='):
            raw_pdf_link = raw_pdf_link[len('/pdf?url='):]
        assert raw_pdf_link.startswith('http')
        pdf_links.append(raw_pdf_link)
        
    # title
    title_soup = parse(get_html_soup(resp.content), parse_config['title'])
    titles = [soup.text for soup in title_soup]
    
    assert len(kimi_links) == len(pdf_links) == len(titles)
    return kimi_links, pdf_links, titles


kimi_links, pdf_links, titles = parse_kimi_links(resp)
print(len(kimi_links), kimi_links[:10])
print(len(pdf_links), pdf_links[:10])
print(len(titles), titles[:10])

1809 ['https://papers.cool/venue/kimi?paper=GXtmuiVrOM@OpenReview', 'https://papers.cool/venue/kimi?paper=ulaUJFd96G@OpenReview', 'https://papers.cool/venue/kimi?paper=liuqDwmbQJ@OpenReview', 'https://papers.cool/venue/kimi?paper=hCrFG9cyuC@OpenReview', 'https://papers.cool/venue/kimi?paper=LNLjU5C5dK@OpenReview', 'https://papers.cool/venue/kimi?paper=MrYiwlDRQO@OpenReview', 'https://papers.cool/venue/kimi?paper=xhCZD9hiiA@OpenReview', 'https://papers.cool/venue/kimi?paper=VXDPXuq4oG@OpenReview', 'https://papers.cool/venue/kimi?paper=Qa0ULgosc9@OpenReview', 'https://papers.cool/venue/kimi?paper=JYu5Flqm9D@OpenReview']
1809 ['https://openreview.net/pdf?id=GXtmuiVrOM', 'https://openreview.net/pdf?id=ulaUJFd96G', 'https://openreview.net/pdf?id=liuqDwmbQJ', 'https://openreview.net/pdf?id=hCrFG9cyuC', 'https://openreview.net/pdf?id=LNLjU5C5dK', 'https://openreview.net/pdf?id=MrYiwlDRQO', 'https://openreview.net/pdf?id=xhCZD9hiiA', 'https://openreview.net/pdf?id=VXDPXuq4oG', 'https://openrev

In [36]:
import aiohttp
from tqdm import tqdm

for i in tqdm(range(100000)):
    async with aiohttp.request('get', "https://papers.cool", proxy='http://10.176.52.116:7890') as resp:
        with open('cookies.txt', 'a') as f:
            f.write(str({'client_id': dict(resp.cookies)['client_id'].value})+'\n')

  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 109/100000 [01:01<15:40:26,  1.77it/s]


CancelledError: 

In [21]:
import requests
from tqdm import tqdm

for i in tqdm(range(100000)):
    resp = requests.get('https://papers.cool', proxies=default_proxy_config)
    # print(resp)
    with open('cookies.txt', 'a') as f:
        f.write(dict(resp.cookies)['client_id']+'\n')
        
        


  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 68/100000 [00:24<9:49:46,  2.82it/s] 


KeyboardInterrupt: 

In [13]:
import requests
cookies = {
    'client_id': "!P3n0PpTkvLxE77zUkxZw6Q==?gAWVKgAAAAAAAACMCWNsaWVudF9pZJSMGDM3MDAzLTE3MTE5NjE1NzguNjY3NDIzMpSGlC4=",
}
resp = requests.post('https://papers.cool/arxiv/kimi?paper=2404.01598', proxies=default_proxy_config, cookies=cookies)

print(resp)
print(resp.headers['content-type'])
with open('kimi_paper.html', 'wb') as f:
    f.write(resp.content)

<Response [200]>
text/html; charset=utf-8
